In [1]:
import pygame
from classes.logic import Card as card
from classes.logic import Player as player
from classes.logic import Game as game
from classes.logic import Deck as deck
from functions import *
from assets import *
import random
import copy

c:\Users\elisk\AppData\Local\Programs\Python\Python313\Lib\site-packages\pygame\pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists


pygame 2.6.1 (SDL 2.28.4, Python 3.13.7)
Hello from the pygame community. https://www.pygame.org/contribute.html


In [2]:
players = ["AI těžké", "AI snadné", "AI snadné", "AI snadné", "AI snadné"]

In [3]:
g = game.Game(players)
for i in range(4):
    g.season = random.sample(g.seasondeck, 1)[0]
    g.seasondeck.remove(g.season)
    for p in g.players:
        p.season_buff = g.s_ref[g.season]['akce']
    g.hands = []
    for i in range(len(g.players)):
        new_hand = deck.PlayerDeck(i, g.game_deck, g.reference)
        for j in new_hand.cardids:
            g.game_deck.remove(j)
        g.hands.append(new_hand)
    for j in range(8):
        for p in range(len(g.players)):
                person = g.players[p]
                g.current_player = p
                g.current_deck = (g.current_player+g.firstplayerdeck)%len(g.players)
                d = g.hands[g.current_deck]
                person.start_turn(d, g)
                info = person.choose(g)
                while info[0]!="end_turn":
                    id = info[2]
                    if info[0]=="store_hand":
                            d.store_card(id, person, g)
                    else:
                        if info[0]=="play_stored":
                            d = person.stored
                        karta = d.cards[id]
                        if karta.card_type == "monster" and karta.cards>0:
                                deck.sample_cards(g, person, karta.cards)
                        msg = d.play_card(id, person, g)
                        if msg == 'fialova':
                            person.play_purple(karta)
                        if msg == 'biom':
                            if len(info)>1:
                                person.play_biom_e(karta, info[2])
                    d = g.hands[g.current_deck]
                    info = person.choose(g)
                person.had_played = False
        g.turn += 1
        g.firstplayerdeck = (g.firstplayerdeck+len(g.hands)+(-1)**g.round)%len(g.players)
        g.current_deck = (g.current_player+g.firstplayerdeck)%len(g.players)
    dice = random.sample([0,0,0,0,1,2], 1)[0] + random.sample([0,0,0,0,1,2], 1)[0]
    goals = []
    for play in range(len(g.players)):
            vals = g.players[play].end_season(dice,  g.s_ref[g.season]['akce'])
            goals.append(vals[5])
    if goals.count(max(goals))==1:
            index = goals.index(max(goals))
            g.players[index].seasons_won +=1
    for person in g.players:
                person.monsters.cards.extend(person.played.cards)
                person.played.cards = []
                person.occupied ={"modra":[], "cerna":[], "hneda":[], "zelena":[], "zlata":[], "fialova":[]}
                for mct in g.mcts:
                    mcts = g.players[mct]
                    mcts.seen = [-1]*len(g.players)
                for biom in person.bioms:
                    person.bioms[biom][0] += person.bioms[biom][1]
                    person.bioms[biom][1] = 0
    
g.end_game()
print(g.order)



IndexError: list index out of range